# 12 — Meta Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## Pattern 1: Prompt Generation

In [3]:
task_description = "I want to analyze customer feedback to identify product improvement opportunities."

meta_prompt = (
    "You are a prompt engineering expert.\n"
    f"Write an effective system prompt for an LLM that will:\n{task_description}\n\n"
    "The prompt should specify:\n"
    "- Role and expertise\n"
    "- Analysis framework\n"
    "- Output format\n"
    "- Constraints\n\n"
    "Return only the system prompt text."
)
generated_prompt = chat([{"role":"user","content":meta_prompt}], temperature=0.3, max_tokens=300)
print("Generated system prompt:")
print(generated_prompt.strip())


Generated system prompt:
"As a seasoned product manager with 8+ years of experience in customer-centric product development, analyze the following customer feedback to identify product improvement opportunities using the Kano Model and the Six Thinking Hats framework. 

Output your analysis in a concise, 3-column table format, with the following columns: 

- **Opportunity**: A clear description of the improvement opportunity identified
- **Category**: The Kano Model category (Basic, Performance, or Excitement) and the Six Thinking Hats perspective (White, Red, Black, Yellow, Green, or Blue) that informed the opportunity
- **Priority**: A numerical priority score (1-5) indicating the relative importance of the opportunity

Assume a product with the following characteristics: 
- Target market: B2C e-commerce
- Product type: Mid-range electronics
- Customer base: 100,000+ users

Analyze 10 customer feedback samples, provided below, and provide your analysis within 500 words. 

Customer Fe

In [4]:
sample_feedback = "The app crashes when I try to upload photos. Also the navigation is confusing. Love the new dark mode though!"
result = system_chat(
    system=generated_prompt,
    user=f"Analyze this feedback: '{sample_feedback}'",
    temperature=0, max_tokens=200
)
print("Analysis using generated prompt:")
print(result.strip())


Analysis using generated prompt:
**Opportunity** | **Category** | **Priority**
-------------------|-------------|-------------
**Improve app stability** | Basic (White) | 5
**Enhance navigation** | Performance (Yellow) | 4
**Add more design options** | Excitement (Blue) | 3

Analysis:

1. **Improve app stability**: The customer experienced a critical issue with the app crashing, which indicates a basic need not being met. This is a high-priority issue as it affects the customer's ability to use the app. (Category: Basic, White; Priority: 5)
2. **Enhance navigation**: The customer found the navigation confusing, which suggests that the app's performance could be improved. This is a moderate-priority issue as it affects the customer's experience but does not prevent them from using the app. (Category: Performance, Yellow; Priority: 4)
3. **Add more design options**: The customer expressed appreciation for the


## Pattern 2: Prompt Improvement

In [5]:
weak_prompt = "Explain blockchain to me"

improve_prompt = (
    "You are a prompt engineering expert.\n"
    f"Improve this prompt to get a better response from an LLM:\n\n"
    f"Original prompt: \"{weak_prompt}\"\n\n"
    "Improved prompt (consider: audience, format, constraints, context):"
)
improved = chat([{"role":"user","content":improve_prompt}], temperature=0, max_tokens=150)
print("Original:", weak_prompt)
print()
print("Improved:")
print(improved.strip())


Original: Explain blockchain to me

Improved:
To improve the prompt and get a better response from an LLM, I would suggest the following:

**Improved Prompt:** "Provide a concise, 2-3 paragraph explanation of blockchain technology, assuming the reader has a basic understanding of computer systems and the internet. Please use simple, non-technical language and avoid jargon. Consider explaining the key components, benefits, and potential applications of blockchain."

**Changes Made:**

1. **Added specificity:** By specifying the desired length of the response (2-3 paragraphs), we can guide the LLM to provide a more focused and concise explanation.
2. **Defined the audience:** By assuming the reader has a basic understanding of computer systems and the internet, we can tailor the explanation to a more


In [6]:
r_weak     = chat([{"role":"user","content":weak_prompt}], temperature=0, max_tokens=100)
r_improved = chat([{"role":"user","content":improved.strip()}], temperature=0, max_tokens=150)

print("Weak prompt response:")
print(r_weak.strip())
print()
print("Improved prompt response:")
print(r_improved.strip())


Weak prompt response:
**What is Blockchain?**

Blockchain is a decentralized, digital ledger technology that allows multiple parties to record and verify transactions without the need for a central authority. It's a chain of blocks, each containing a set of transactions, that are linked together through cryptography.

**Key Components of Blockchain:**

1. **Decentralized Network**: A network of computers (nodes) that work together to validate and record transactions.
2. **Digital Ledger**: A digital book that contains a record of all transactions.

Improved prompt response:
Your suggested improvements are excellent ways to refine the prompt and elicit a more effective response from a Large Language Model (LLM). By adding specificity, defining the audience, and providing clear guidelines, you can help the LLM generate a more targeted and engaging explanation.

Here's a breakdown of the changes you made and their benefits:

1. **Added specificity:** By specifying the desired length of th

## Meta Prompting for Templates

In [7]:
template_request = (
    "Create a reusable prompt template for summarizing research papers.\n"
    "The template should have placeholders like {title}, {content}, {audience}.\n"
    "Include instructions for length, format, and key elements to extract."
)
template = chat([{"role":"user","content":template_request}], temperature=0, max_tokens=250)
print("Generated template:")
print(template.strip())


Generated template:
**Reusable Prompt Template for Summarizing Research Papers**

**Summary Template:**

**Title:** {title}
**Summary:** 
{content} 
**Target Audience:** {audience}

**Instructions:**

1. **Length:** The summary should be approximately 150-200 words in length.
2. **Format:** Use a clear and concise writing style, avoiding technical jargon and complex terminology.
3. **Key Elements to Extract:**
	* **Research Question:** What is the main research question or hypothesis being investigated?
	* **Methodology:** What methods were used to collect and analyze data?
	* **Key Findings:** What are the most significant results or conclusions drawn from the study?
	* **Implications:** What are the implications of the study for the field or industry?
	* **Limitations:** What are the limitations of the study, and how might they impact the results?
4. **Tone and Style:**
	* Use a neutral tone, avoiding bias or opinion.
	* Focus on conveying the main ideas and findings of the study.
	*

In [8]:
filled = template.strip()
filled = filled.replace("{title}", "Attention Is All You Need")
filled = filled.replace("{content}", "A transformer architecture using self-attention mechanisms, proposed by Vaswani et al. 2017, that removes recurrence entirely.")
filled = filled.replace("{audience}", "software engineers new to ML")

result = chat([{"role":"user","content":filled}], temperature=0, max_tokens=150)
print("Filled template result:")
print(result.strip())


Filled template result:
**Summary Template:**

**Title:** Attention Is All You Need
**Summary:**
The paper "Attention Is All You Need" by Vaswani et al. (2017) introduces a transformer architecture that uses self-attention mechanisms to process sequential data. This approach eliminates the need for recurrence, making it a significant departure from traditional recurrent neural networks (RNNs).

**Target Audience:** software engineers new to ML

**Summary:**

The researchers aimed to investigate whether a transformer architecture could outperform RNNs in language translation tasks. To achieve this, they designed a model that uses self-attention mechanisms to weigh the importance of different input elements. The model was trained on a large dataset of parallel sentences and evaluated on a benchmark translation
